In [63]:
from cmdstanpy import CmdStanModel
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm
import traceback
import ast

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Optional

import matplotlib.pyplot as plt

### Compile Stan model

In [31]:
stan_path = Path("/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/stan/differential_dosage_model_v1.stan")
model = CmdStanModel(stan_file=str(stan_path))

#### Fit one gene / multiple genes - CRC dataset

In [33]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/CRC/data/stan_model_test/"
df_long = pd.read_csv(os.path.join(DATA_PATH, "crc_joint_long_de_drivers.csv"))
df_long.head()

,gene,sampleID,expr,copies,subtype,purity,ploidy,sf
0,ACVR1B,CRC.SW.U0001.T,7593,2,MSS,0.41,3.8,1.293201
1,ACVR1B,CRC.SW.U0002.T,5003,2,MSS,0.37,3.3,1.024008
2,ACVR1B,CRC.SW.U0004.T,4994,2,MSS,0.58,3.2,1.169870
3,ACVR1B,CRC.SW.U0030.T,6231,2,MSS,0.59,2.2,1.290047
4,ACVR1B,CRC.SW.U0066.T,2908,2,MSS,0.41,1.9,0.971163


In [35]:
# Select one gene
gene_df = df_long[df_long["gene"] == "BRAF"]
gene_df.head()

,gene,sampleID,expr,copies,subtype,purity,ploidy,sf
19720,BRAF,CRC.SW.U0001.T,3100,2,MSS,0.41,3.8,1.293201
19721,BRAF,CRC.SW.U0002.T,2913,3,MSS,0.37,3.3,1.024008
19722,BRAF,CRC.SW.U0004.T,2028,2,MSS,0.58,3.2,1.169870
19723,BRAF,CRC.SW.U0030.T,2008,2,MSS,0.59,2.2,1.290047
19724,BRAF,CRC.SW.U0066.T,2066,2,MSS,0.41,1.9,0.971163


In [37]:
# Select multiple genes
#gene_df = df_long[df_long["gene"].isin(["ZNRF3", "SMAD4", "APC", "LILRB3"])]
#gene_df.head()

gene_df = df_long[df_long["gene"].isin(["ASXL1", "ERBB2", "SMAD4", "BCOR", "JUN"])]
gene_df.head()

,gene,sampleID,expr,copies,subtype,purity,ploidy,sf
10846,ASXL1,CRC.SW.U0001.T,9578,4,MSS,0.41,3.8,1.293201
10847,ASXL1,CRC.SW.U0002.T,6899,3,MSS,0.37,3.3,1.024008
10848,ASXL1,CRC.SW.U0004.T,7049,4,MSS,0.58,3.2,1.169870
10849,ASXL1,CRC.SW.U0030.T,5707,3,MSS,0.59,2.2,1.290047
10850,ASXL1,CRC.SW.U0066.T,5014,2,MSS,0.41,1.9,0.971163


### Model Simulator

In [ ]:
# -----------------------------------------
# 1. SAMPLE-LEVEL COVARIATES
# -----------------------------------------
def simulate_covariates(
    N=200,
    subtype_levels=("A", "B"),
    seed=123,
    copy_probs=(0.15, 0.50, 0.15, 0.10, 0.10),
    purity_a=5.0,
    purity_b=2.0,
    sf_meanlog=0.0,
    sf_sdlog=0.2,
):
    """
    Simulate sample-level covariates shared by all genes.

    Returns
    -------
    covars : dict
        Contains N, S, subtype labels/codes, sf, purity, copies, dose_log, dev.
    """
    rng = np.random.default_rng(seed)
    subtype_levels = tuple(subtype_levels)
    S = len(subtype_levels)

    # Balanced-ish subtype assignment
    subtype_idx = rng.integers(low=0, high=S, size=N)   # 0..S-1
    subtype_labels = np.array(subtype_levels)[subtype_idx]

    # Library size factors around 1
    sf = rng.lognormal(mean=sf_meanlog, sigma=sf_sdlog, size=N)

    # Tumor purity
    purity = rng.beta(a=purity_a, b=purity_b, size=N)

    # Copy number
    copies = rng.choice(
        [1, 2, 3, 4, 5],
        size=N,
        p=np.asarray(copy_probs, dtype=float)
    ).astype(float)

    CN_eff = np.clip(copies, 1.0, None)
    dose_log = np.log(CN_eff / 2.0)
    dev = (copies - 2.0) / 2.0

    return {
        "N": N,
        "S": S,
        "subtype_levels": subtype_levels,
        "subtype_idx": subtype_idx,
        "subtype_labels": subtype_labels,
        "sf": sf,
        "purity": purity,
        "copies": copies,
        "dose_log": dose_log,
        "dev": dev,
    }


# ---------------------------------------------------
# 2. TRUE PARAMETER GENERATION BY SCENARIO
# ---------------------------------------------------

def _sum_to_zero_offsets(rng, S, sd):
    x = rng.normal(0.0, sd, size=S)
    return x - x.mean()


def simulate_gene_params(
    covars,
    rng,
    gene_id,
    scenario="mixed",
    b0_mean_loc=5.0,
    b0_mean_sd=0.7,
    b0_offset_sd=0.3,
    b_scaling_offset_sd=0.10,
    b_dev_offset_sd=0.08,
    b_noncancer_loc=np.log(5.0),
    b_noncancer_sd=0.5,
    phi_logmean=np.log(10.0),
    phi_logsd=0.25,
):
    """
    Simulate true parameters for one gene under an explicit scenario.

    Scenarios
    ---------
    null
        No CN effect.
    scaling
        Positive dosage scaling, no deviation.
    deviation_pos
        No scaling, positive deviation.
    deviation_neg
        No scaling, negative deviation.
    mixed
        Positive scaling with negative deviation (partial compensation).
    """

    S = covars["S"]

    # Tumor baseline
    b0_mean = rng.normal(b0_mean_loc, b0_mean_sd)
    b0 = b0_mean + _sum_to_zero_offsets(rng, S, b0_offset_sd)

    # Scenario-specific global means
    if scenario == "null":
        b_scaling_mean = rng.normal(0.0, 0.05)
        b_dev_mean = rng.normal(0.0, 0.05)

    elif scenario == "scaling":
        b_scaling_mean = rng.normal(0.60, 0.10)
        b_dev_mean = rng.normal(0.0, 0.05)

    elif scenario == "deviation_pos":
        b_scaling_mean = rng.normal(0.0, 0.05)
        b_dev_mean = rng.normal(0.40, 0.10)

    elif scenario == "deviation_neg":
        b_scaling_mean = rng.normal(0.0, 0.05)
        b_dev_mean = rng.normal(-0.40, 0.10)

    elif scenario == "mixed":
        b_scaling_mean = rng.normal(0.50, 0.10)
        b_dev_mean = rng.normal(-0.25, 0.10)

    else:
        raise ValueError(
            "scenario must be one of: "
            "'null', 'scaling', 'deviation_pos', 'deviation_neg', 'mixed'"
        )

    # Subtype-specific deviations around global means
    b_scaling = b_scaling_mean + _sum_to_zero_offsets(rng, S, b_scaling_offset_sd)
    b_deviation = b_dev_mean + _sum_to_zero_offsets(rng, S, b_dev_offset_sd)

    # Non-tumor baseline
    b_noncancer_log = rng.normal(b_noncancer_loc, b_noncancer_sd)

    # NB dispersion
    phi = float(np.exp(rng.normal(phi_logmean, phi_logsd)))

    return {
        "gene": gene_id,
        "scenario": scenario,
        "b0": b0,
        "b_scaling": b_scaling,
        "b_deviation": b_deviation,
        "b_noncancer_log": b_noncancer_log,
        "phi": phi,
        "b0_mean": b0_mean,
        "b_scaling_mean": b_scaling_mean,
        "b_dev_mean": b_dev_mean,
    }


# --------------------------------------
# 3. COUNT GENERATION
# --------------------------------------

def simulate_counts_for_gene(covars, true_params, rng):
    """
    Simulate NB2 counts under the same mean structure as the Stan model.
    """

    N = covars["N"]
    subtype_idx = covars["subtype_idx"]
    sf = covars["sf"]
    purity = covars["purity"]
    dose_log = covars["dose_log"]
    dev = covars["dev"]

    b0 = true_params["b0"]
    b_scaling = true_params["b_scaling"]
    b_deviation = true_params["b_deviation"]
    b_noncancer_log = true_params["b_noncancer_log"]
    phi = true_params["phi"]

    mu = np.empty(N, dtype=float)

    for n in range(N):
        s = subtype_idx[n]  # 0-based subtype index

        linpred = (
            b0[s]
            + dose_log[n] * b_scaling[s]
            + dev[n] * b_deviation[s]
        )

        tumor_mu = sf[n] * purity[n] * np.exp(linpred)
        stroma_mu = sf[n] * (1.0 - purity[n]) * np.exp(b_noncancer_log)
        mu[n] = tumor_mu + stroma_mu

    # NB2(mu, phi) in NumPy parameterization
    # Var(y) = mu + mu^2 / phi
    r = phi
    p = r / (r + mu)
    p = np.clip(p, 1e-10, 1 - 1e-10)

    expr = rng.negative_binomial(n=r, p=p, size=N).astype(int)

    return expr, mu


# --------------------------------------
# 4. SCENARIO ASSIGNMENT
# --------------------------------------

def assign_gene_scenarios(
    G,
    scenario_probs=None,
    rng=None,
):
    """
    Assign a scenario to each gene.

    scenario_probs example:
    {
        "null": 0.25,
        "scaling": 0.25,
        "deviation_pos": 0.20,
        "deviation_neg": 0.10,
        "mixed": 0.20,
    }
    """
    if rng is None:
        rng = np.random.default_rng(123)

    if scenario_probs is None:
        scenario_probs = {
            "null": 0.30,
            "scaling": 0.30,
            "deviation_pos": 0.10,
            "deviation_neg": 0.10,
            "mixed": 0.20,
        }

    scenarios = list(scenario_probs.keys())
    probs = np.array(list(scenario_probs.values()), dtype=float)
    probs = probs / probs.sum()

    return rng.choice(scenarios, size=G, p=probs)

### Model fit function

In [39]:
# more diagnostic parameters + MCMC + VI

def fit_one_gene_de(
    gene_df: pd.DataFrame,
    model: "CmdStanModel",
    gene: str | None = None,
    cna: str = "all",
    et: float = 0.15,
    min_aneup: int = 5,
    min_unique_counts: int = 5,
    min_cn_abs_sum: float = 1.0,
    subtype_col: str = "subtype",
    subtype_order: list[str] | None = None,
    chains: int = 4,
    iter_warmup: int = 1000,
    iter_sampling: int = 1000,
    seed: int = 1,
    show_progress: bool = False,
    adapt_delta: float = 0.9,
    max_treedepth: int = 12,
    rope_logfc: float = float(np.log(1.2)),
    eps_frac: float = 0.10,          # ROPE on fracCN for ~10% change
    return_all_subtypes: bool = True,
    engine: str = "nuts",               # "nuts", "vi_meanfield", "vi_fullrank"
    vi_iter: int = 20000,
    vi_output_samples: int = 2000,
    vi_elbo_samples: int = 100,
    vi_grad_samples: int = 1,
    return_ppc: bool = False,
    save_ppc_draws: bool = False,
    ppc_thin: int = 10,
    save_draws: bool = False,
    output_dir: str | Path | None = None,
):
    """
    Fit the Bayesian differential gene-dosage model for a single gene.

    Required columns in gene_df:
        expr, copies, purity, sf, subtype_col
    Optional:
        gene (if gene_df contains multiple genes)

    Stan model is the log-link CN-expression model with generated quantities:
        delta_tumor0_log, delta_scaling, delta_dev,
        lp_2to1[s], lp_2to3[s], lp_2to4[s],
        (and optionally fracCN_*, cancel_index_*, p_DC_* if you added them).
    """

    # ---------- helpers ----------
    def q(x: np.ndarray) -> list[float]:
        return np.quantile(x, [0.025, 0.5, 0.975]).tolist()

    def summarize_draw_1d(x: np.ndarray, prefix: str) -> dict:
        qi = q(x)
        return {
            f"{prefix}_median": float(qi[1]),
            f"{prefix}_q025": float(qi[0]),
            f"{prefix}_q975": float(qi[2]),
        }

    def summarize_draw_2d(arr_2d: np.ndarray, s_idx0: int, prefix: str) -> dict:
        # arr_2d: (draws, S), s_idx0: 0-based subtype index
        return summarize_draw_1d(arr_2d[:, s_idx0], prefix)

    # ---------- output dir ----------
    out_dir = None
    if output_dir is not None:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

    df = gene_df.copy()

    # ---- subset gene ----
    if gene is not None and "gene" in df.columns and df["gene"].nunique() > 1:
        df = df.loc[df["gene"] == gene].copy()
    if gene is None and "gene" in df.columns and df["gene"].nunique() == 1:
        gene = str(df["gene"].iloc[0])

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_rows_for_gene"}

    required = {"expr", "copies", "purity", "sf", subtype_col}
    missing = required - set(df.columns)
    if missing:
        return {"status": "error", "gene": gene, "reason": f"missing_columns: {sorted(missing)}"}

    # ---- CNA subset ----
    if cna == "amp":
        df = df[df["copies"] > (2 - et)]
    elif cna == "del":
        df = df[df["copies"] < (2 + et)]
    elif cna == "all":
        pass
    else:
        raise ValueError("cna must be 'amp', 'del', or 'all'")

    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "no_samples_after_cna_filter"}

    # ---- QC ----
    df = df.dropna(subset=list(required))
    if df.empty:
        return {"status": "skipped", "gene": gene, "reason": "all_na_after_dropna"}

    if (df["expr"] < 0).any():
        return {"status": "error", "gene": gene, "reason": "negative_counts"}

    if not df["purity"].between(0, 1).all():
        return {"status": "error", "gene": gene, "reason": "purity_out_of_bounds"}

    if not (df["sf"] > 0).all():
        return {"status": "error", "gene": gene, "reason": "nonpositive_sf"}

    if df["expr"].nunique() < min_unique_counts:
        return {"status": "skipped", "gene": gene, "reason": "too_few_unique_counts"}

    # aneuploid count check
    n_aneup = int((np.abs(df["copies"].astype(float) - 2.0) > (1.0 - et)).sum())
    if n_aneup < min_aneup or (df["expr"] == 0).all():
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "low_aneup_or_all_zero",
        }

    # identifiability check: need some CN deviation mass
    dev_tmp = (df["copies"].astype(float) - 2.0) / 2.0
    if cna == "all" and float(np.abs(dev_tmp).sum()) < min_cn_abs_sum:
        return {
            "status": "skipped",
            "gene": gene,
            "n_aneup": n_aneup,
            "reason": "too_little_cn_variation",
        }

    # ---- subtype encoding -----
    if subtype_order is not None:
        cat = pd.Categorical(df[subtype_col], categories=subtype_order, ordered=True)
        if cat.isna().any():
            bad = df.loc[cat.isna(), subtype_col].unique().tolist()
            return {
                "status": "error",
                "gene": gene,
                "reason": f"unknown_subtypes: {bad}",
            }
        subtype_codes = (cat.codes + 1).astype(int)
        levels = list(cat.categories)
    else:
        levels = sorted(pd.unique(df[subtype_col]).tolist())
        mapping = {lv: i + 1 for i, lv in enumerate(levels)}
        subtype_codes = df[subtype_col].map(mapping).astype(int).to_numpy()

    S = len(levels)
    if S < 2:
        return {
            "status": "skipped",
            "gene": gene,
            "reason": "need_at_least_2_subtypes_present_for_DE",
        }

    # ---- CN covariates (identified) ----
    # effective CN for scaling: treat 0 and 1 as "1 copy"
    CN_eff = df["copies"].astype(float).clip(lower=1.0)
    #df["dose_log"] = np.log(np.maximum(df["copies"].astype(float), 0.1) / 2.0)
    df["dose_log"] = np.log(CN_eff / 2.0)
    df["dev"] = (df["copies"].astype(float) - 2.0) / 2.0

    stan_data = {
        "N": int(len(df)),
        "y": df["expr"].astype(int).to_numpy(),
        "S": int(S),
        "subtype": np.asarray(subtype_codes, dtype=int),
        "sf": df["sf"].to_numpy(dtype=float),
        "purity": df["purity"].to_numpy(dtype=float),
        "dose_log": df["dose_log"].to_numpy(dtype=float),
        "dev": df["dev"].to_numpy(dtype=float),
    }

    # ---- initial values for NUTS ----
    rng = np.random.default_rng(seed)
    phi_init = rng.exponential(1.0)
    phi_init = float(np.clip(phi_init, 1e-3, 100.0))
    init_nuts = {
        # global means
        "b0_mean": float(rng.normal(0.0, 0.2)),
        "b_scaling_mean": float(rng.normal(0.0, 0.2)),
        "b_dev_mean": float(rng.normal(0.0, 0.05)),
        
        "b0_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_scaling_offset": rng.normal(0.0, 0.1, size=S).tolist(),
        "b_dev_offset": rng.normal(0.0, 0.05, size=S).tolist(),
        
        # stromal baseline
        "b_noncancer_log": float(rng.normal(0.0, 0.2)),
        
        # dispersion 
        "phi": phi_init
    }

    # ---- inference engine ----
    engine = engine.lower()
    using_mcmc = False
    fit = None
    draws_df = None  # only used for VI; for MCMC we extract arrays directly
    
    if engine == "nuts":
        using_mcmc = True
        fit = model.sample(
            data=stan_data,
            chains=chains,
            iter_warmup=iter_warmup,
            iter_sampling=iter_sampling,
            seed=seed,
            inits=init_nuts,
            show_progress=show_progress,
            adapt_delta=adapt_delta,
            max_treedepth=max_treedepth,
        )
        # persist csvs in a predictable place
        if out_dir is not None and hasattr(fit, "save_csvfiles"):
            csv_dir = out_dir / "csv"
            csv_dir.mkdir(exist_ok=True)
            fit.save_csvfiles(dir=str(csv_dir))

    elif engine in {"vi_meanfield", "vi_fullrank"}:
        algo = "meanfield" if engine == "vi_meanfield" else "fullrank"

        # NOTE: CmdStan's variational inits arg is a *float range*, not a dict.
        # We'll just let Stan pick its default random init unless you want to
        # pass a scalar, e.g. inits=0.1.
        fit = model.variational(
            data=stan_data,
            seed=seed,
            algorithm=algo,
            iter=vi_iter,
            grad_samples=vi_grad_samples,
            elbo_samples=vi_elbo_samples,
            output_samples=vi_output_samples,
            show_console=show_progress,
        )
        # VI: build a DataFrame manually
        # variational_sample: array of shape (draws, num_params)
        draws_np = fit.variational_sample
        colnames = fit.column_names
        draws_df = pd.DataFrame(draws_np, columns=colnames)

    else:
        raise ValueError("engine must be 'nuts', 'vi_meanfield', or 'vi_fullrank'")

   # ---------- extract essentials ----------
    if using_mcmc:
        # Fast, memory-safe extraction
        d_tumor = fit.stan_variable("delta_tumor0_log")
        d_scal = fit.stan_variable("delta_scaling")
        d_dev = fit.stan_variable("delta_dev")
        phi_arr = fit.stan_variable("phi")
        b_nc = fit.stan_variable("b_noncancer_log")

        # subtype-specific vectors (draws, S)
        b0 = fit.stan_variable("b0")
        b_scaling = fit.stan_variable("b_scaling")
        b_deviation = fit.stan_variable("b_deviation")

        # canonical transition summaries (draws, S)
        lp_2to1 = fit.stan_variable("lp_2to1")
        lp_2to3 = fit.stan_variable("lp_2to3")
        lp_2to4 = fit.stan_variable("lp_2to4")

        # optional mechanistic decompositions if present
        def maybe_var(name: str):
            try:
                return fit.stan_variable(name)
            except Exception:
                return None

        lp_scaling_2to1 = maybe_var("lp_scaling_2to1")
        lp_dev_2to1 = maybe_var("lp_dev_2to1")
        lp_scaling_2to3 = maybe_var("lp_scaling_2to3")
        lp_dev_2to3 = maybe_var("lp_dev_2to3")
        lp_scaling_2to4 = maybe_var("lp_scaling_2to4")
        lp_dev_2to4 = maybe_var("lp_dev_2to4")

    else:
        # VI: arrays from draws_df
        req = ["delta_tumor0_log", "delta_scaling", "delta_dev", "phi", "b_noncancer_log"]
        missing_cols = [c for c in req if c not in draws_df.columns]
        if missing_cols:
            return {"status": "error", "gene": gene, "reason": f"missing_draws_columns: {missing_cols}"}

        d_tumor = draws_df["delta_tumor0_log"].to_numpy()
        d_scal = draws_df["delta_scaling"].to_numpy()
        d_dev = draws_df["delta_dev"].to_numpy()
        phi_arr = draws_df["phi"].to_numpy()
        b_nc = draws_df["b_noncancer_log"].to_numpy()

        # VI columns are like b0[1], ...; build arrays (draws, S) if present
        def stack_param(base: str) -> np.ndarray | None:
            cols = [f"{base}[{s}]" for s in range(1, S + 1)]
            if all(c in draws_df.columns for c in cols):
                return draws_df[cols].to_numpy()
            return None

        b0 = stack_param("b0")
        b_scaling = stack_param("b_scaling")
        b_deviation = stack_param("b_deviation")

        lp_2to1 = stack_param("lp_2to1")
        lp_2to3 = stack_param("lp_2to3")
        lp_2to4 = stack_param("lp_2to4")

        # optional
        lp_scaling_2to1 = stack_param("lp_scaling_2to1")
        lp_dev_2to1 = stack_param("lp_dev_2to1")
        lp_scaling_2to3 = stack_param("lp_scaling_2to3")
        lp_dev_2to3 = stack_param("lp_dev_2to3")
        lp_scaling_2to4 = stack_param("lp_scaling_2to4")
        lp_dev_2to4 = stack_param("lp_dev_2to4")

    # ---------- main summaries ----------
    p_up_tumor = float((d_tumor > 0).mean())
    lfsr_tumor = float(min(p_up_tumor, 1.0 - p_up_tumor))
    p_rope_tumor = float((np.abs(d_tumor) <= rope_logfc).mean())

    p_up_scal = float((d_scal > 0).mean())
    lfsr_scal = float(min(p_up_scal, 1.0 - p_up_scal))

    p_up_dev = float((d_dev > 0).mean())
    lfsr_dev = float(min(p_up_dev, 1.0 - p_up_dev))

    ln2 = np.log(2.0)
    lfc_tumor = d_tumor / ln2
    lfc_ci = q(lfc_tumor)

    out: dict[str, object] = {
        "status": "ok",
        "gene": gene,
        "N": int(len(df)),
        "n_aneup": n_aneup,
        "cna": cna,
        "subtype_levels": levels,
        "engine": engine,
        "seed": int(seed),
        "output_dir": str(out_dir) if out_dir is not None else None,

        "tumor0_lfc_median": float(lfc_ci[1]),
        "tumor0_lfc_q025": float(lfc_ci[0]),
        "tumor0_lfc_q975": float(lfc_ci[2]),
        "p_up_tumor": p_up_tumor,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,

        "delta_scaling_median": float(q(d_scal)[1]),
        "delta_scaling_q025": float(q(d_scal)[0]),
        "delta_scaling_q975": float(q(d_scal)[2]),
        "p_up_scaling": p_up_scal,
        "lfsr_scaling": lfsr_scal,

        "delta_dev_median": float(q(d_dev)[1]),
        "delta_dev_q025": float(q(d_dev)[0]),
        "delta_dev_q975": float(q(d_dev)[2]),
        "p_up_dev": p_up_dev,
        "lfsr_dev": lfsr_dev,

        "phi_median": float(q(phi_arr)[1]),
        "phi_q025": float(q(phi_arr)[0]),
        "phi_q975": float(q(phi_arr)[2]),
        "b_noncancer_log_median": float(q(b_nc)[1]),
        "b_noncancer_log_q025": float(q(b_nc)[0]),
        "b_noncancer_log_q975": float(q(b_nc)[2]),
    }

    # ---------- subtype-specific summaries ----------
    s_iter = range(1, S + 1) if return_all_subtypes else range(1, min(S, 2) + 1)

    if b0 is not None:
        for s in s_iter:
            s0 = s - 1
            out.update(summarize_draw_2d(b0, s0, f"b0_s{s}"))
    if b_scaling is not None:
        for s in s_iter:
            s0 = s - 1
            out.update(summarize_draw_2d(b_scaling, s0, f"b_scaling_s{s}"))
    if b_deviation is not None:
        for s in s_iter:
            s0 = s - 1
            out.update(summarize_draw_2d(b_deviation, s0, f"b_deviation_s{s}"))

    # transitions
    for s in s_iter:
        s0 = s - 1
        if lp_2to1 is not None:
            lp21 = lp_2to1[:, s0]
            out.update(summarize_draw_1d(lp21, f"lp_2to1_s{s}"))
            frac21 = np.expm1(lp21)
            out.update(summarize_draw_1d(frac21, f"fracCN_2to1_s{s}"))
            out[f"p_fracCN_2to1_pos_s{s}"] = float((frac21 > eps_frac).mean())
            out[f"p_fracCN_2to1_rope_s{s}"] = float((np.abs(frac21) <= eps_frac).mean())
            out[f"p_fracCN_2to1_neg_s{s}"] = float((frac21 < -eps_frac).mean())

        if lp_2to3 is not None:
            lp23 = lp_2to3[:, s0]
            out.update(summarize_draw_1d(lp23, f"lp_2to3_s{s}"))
            frac23 = np.expm1(lp23)
            out.update(summarize_draw_1d(frac23, f"fracCN_2to3_s{s}"))
            out[f"p_fracCN_2to3_pos_s{s}"] = float((frac23 > eps_frac).mean())
            out[f"p_fracCN_2to3_rope_s{s}"] = float((np.abs(frac23) <= eps_frac).mean())
            out[f"p_fracCN_2to3_neg_s{s}"] = float((frac23 < -eps_frac).mean())

        if lp_2to4 is not None:
            lp24 = lp_2to4[:, s0]
            out.update(summarize_draw_1d(lp24, f"lp_2to4_s{s}"))
            frac24 = np.expm1(lp24)
            out.update(summarize_draw_1d(frac24, f"fracCN_2to4_s{s}"))
            out[f"p_fracCN_2to4_pos_s{s}"] = float((frac24 > eps_frac).mean())
            out[f"p_fracCN_2to4_rope_s{s}"] = float((np.abs(frac24) <= eps_frac).mean())
            out[f"p_fracCN_2to4_neg_s{s}"] = float((frac24 < -eps_frac).mean())

        # optional decomposition pieces
        if lp_scaling_2to1 is not None:
            out.update(summarize_draw_1d(lp_scaling_2to1[:, s0], f"lp_scaling_2to1_s{s}"))
        if lp_dev_2to1 is not None:
            out.update(summarize_draw_1d(lp_dev_2to1[:, s0], f"lp_dev_2to1_s{s}"))

        if lp_scaling_2to3 is not None:
            out.update(summarize_draw_1d(lp_scaling_2to3[:, s0], f"lp_scaling_2to3_s{s}"))
        if lp_dev_2to3 is not None:
            out.update(summarize_draw_1d(lp_dev_2to3[:, s0], f"lp_dev_2to3_s{s}"))

        if lp_scaling_2to4 is not None:
            out.update(summarize_draw_1d(lp_scaling_2to4[:, s0], f"lp_scaling_2to4_s{s}"))
        if lp_dev_2to4 is not None:
            out.update(summarize_draw_1d(lp_dev_2to4[:, s0], f"lp_dev_2to4_s{s}"))

    
    # ---------- PPC ----------
    if return_ppc and using_mcmc:
        y_obs = stan_data["y"]
        y_rep = fit.stan_variable("y_rep")  # (draws, N)

        out["ppc_summary"] = {
            "obs_mean": float(y_obs.mean()),
            "obs_var": float(y_obs.var(ddof=1)),
            "obs_zero_frac": float((y_obs == 0).mean()),
            "ppc_mean_median": float(np.median(y_rep.mean(axis=1))),
            "ppc_var_median": float(np.median(y_rep.var(axis=1, ddof=1))),
            "ppc_zero_frac_median": float(np.median((y_rep == 0).mean(axis=1))),
        }

        if save_ppc_draws and out_dir is not None:
            y_rep_thin = y_rep[:: max(1, int(ppc_thin)), :]
            ppc_path = out_dir / "ppc_y_rep_thin.npz"
            np.savez_compressed(ppc_path, y_rep=y_rep_thin, y_obs=y_obs)
            out["ppc_path"] = str(ppc_path)

    # ---------- PPC ----------
    if return_ppc and using_mcmc:
        y_obs = stan_data["y"]
        y_rep = fit.stan_variable("y_rep")  # (draws, N)

        out["ppc_summary"] = {
            "obs_mean": float(y_obs.mean()),
            "obs_var": float(y_obs.var(ddof=1)),
            "obs_zero_frac": float((y_obs == 0).mean()),
            "ppc_mean_median": float(np.median(y_rep.mean(axis=1))),
            "ppc_var_median": float(np.median(y_rep.var(axis=1, ddof=1))),
            "ppc_zero_frac_median": float(np.median((y_rep == 0).mean(axis=1))),
        }

        if save_ppc_draws and out_dir is not None:
            y_rep_thin = y_rep[:: max(1, int(ppc_thin)), :]
            ppc_path = out_dir / "ppc_y_rep_thin.npz"
            np.savez_compressed(ppc_path, y_rep=y_rep_thin, y_obs=y_obs)
            out["ppc_path"] = str(ppc_path)

    # ---------- sampler diagnostics ----------
    if using_mcmc:
        # summary CSV (quick human inspection)
        try:
            summ = fit.summary()
            if out_dir is not None:
                summ_path = out_dir / "summary.csv"
                summ.to_csv(summ_path, index=True)
                out["summary_csv"] = str(summ_path)

            # Rhat / ESS core
            rhat_col = next((c for c in ["R_hat", "Rhat"] if c in summ.columns), None)
            ess_col = next((c for c in ["Ess_bulk", "ESS_bulk", "N_Eff", "Ess"] if c in summ.columns), None)

            core_params = ["delta_tumor0_log", "delta_scaling", "delta_dev", "phi"]
            for s in s_iter:
                core_params += [f"b0[{s}]", f"b_scaling[{s}]", f"b_deviation[{s}]"]
            core_params = [p for p in core_params if p in summ.index]

            if rhat_col and core_params:
                rhat_vals = summ.loc[core_params, rhat_col].to_numpy(dtype=float)
                out["max_Rhat_core"] = float(np.nanmax(rhat_vals))
            else:
                out["max_Rhat_core"] = float("nan")

            if ess_col and core_params:
                ess_vals = summ.loc[core_params, ess_col].to_numpy(dtype=float)
                out["min_ESS_core"] = float(np.nanmin(ess_vals))
            else:
                out["min_ESS_core"] = float("nan")

        except Exception:
            out["summary_csv"] = None
            out["max_Rhat_core"] = float("nan")
            out["min_ESS_core"] = float("nan")

        # diagnose text (contains divergences / treedepth warnings)
        try:
            diag = fit.diagnose()
            out["diagnose"] = diag
            if out_dir is not None:
                (out_dir / "diagnose.txt").write_text(diag)
        except Exception:
            out["diagnose"] = None

        # simple flag
        out["fit_flag"] = "ok"
        if (np.isfinite(out.get("max_Rhat_core", np.nan)) and out["max_Rhat_core"] > 1.05) or (
            np.isfinite(out.get("min_ESS_core", np.nan)) and out["min_ESS_core"] < 100
        ):
            out["fit_flag"] = "warn"

    else:
        out["summary_csv"] = None
        out["max_Rhat_core"] = float("nan")
        out["min_ESS_core"] = float("nan")
        out["diagnose"] = None
        out["fit_flag"] = "ok"

    # ---------- optional: save a compact subset of draws ----------
    if save_draws:
        # Save small NPZ with only core arrays (much smaller than draws_pd)
        if out_dir is not None and using_mcmc:
            keep_path = out_dir / "draws_subset.npz"
            np.savez_compressed(
                keep_path,
                delta_tumor0_log=d_tumor,
                delta_scaling=d_scal,
                delta_dev=d_dev,
                phi=phi_arr,
                b_noncancer_log=b_nc,
                b0=b0 if b0 is not None else np.array([]),
                b_scaling=b_scaling if b_scaling is not None else np.array([]),
                b_deviation=b_deviation if b_deviation is not None else np.array([]),
            )
            out["draws_subset_path"] = str(keep_path)
        else:
            out["draws_subset_path"] = None

    return out

#### Fit on simulated data

In [31]:
def run_gene(gene_id, data_df,seed_base=1):
    """
    #Fit one gene and return the result dict from fit_one_gene_de.
    """
    df_g = data_df[data_df["gene"] == gene_id].copy()

    # Optional: give each gene a different seed for reproducibility
    # (and so chains are different across genes)
    seed = seed_base + hash(gene_id) % 10_000_000

    res = fit_one_gene_de(
        df_g,
        model,
        gene=gene_id,
        engine="nuts",
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=seed,
        show_progress=False,
        adapt_delta=0.9,
        max_treedepth=12,
    )
    return res

genes = gene_df["gene"].unique()

results = Parallel(n_jobs=4, backend="loky")(
    delayed(run_gene(g, gene_df)) for g in genes
)

results_df = pd.DataFrame(results)

In [13]:
# MCMC
res_mcmc = fit_one_gene_de(
        gene_df,
        model,
        gene="BRAF",
        engine="nuts",
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=123,
        show_progress=False,
        adapt_delta=0.9,
        max_treedepth=12,
        return_ppc=True
    )

21:42:25 - cmdstanpy - INFO - CmdStan start processing
21:42:25 - cmdstanpy - INFO - Chain [1] start processing
21:42:25 - cmdstanpy - INFO - Chain [2] start processing
21:42:25 - cmdstanpy - INFO - Chain [3] start processing
21:42:25 - cmdstanpy - INFO - Chain [4] start processing
21:43:36 - cmdstanpy - INFO - Chain [4] done processing
21:43:43 - cmdstanpy - INFO - Chain [3] done processing
21:43:43 - cmdstanpy - INFO - Chain [1] done processing
21:43:46 - cmdstanpy - INFO - Chain [2] done processing


In [15]:
res_mcmc

{'status': 'ok',
 'gene': 'BRAF',
 'N': 986,
 'n_aneup': 139,
 'cna': 'all',
 'subtype_levels': ['MSI', 'MSS'],
 'engine': 'nuts',
 'seed': 123,
 'output_dir': None,
 'tumor0_lfc_median': 0.5011424842259152,
 'tumor0_lfc_q025': 0.40585990665465976,
 'tumor0_lfc_q975': 0.6046302455727227,
 'p_up_tumor': 1.0,
 'lfsr_tumor': 0.0,
 'p_rope_tumor': 0.0,
 'delta_scaling_median': -0.26872799999999997,
 'delta_scaling_q025': -0.8261439500000001,
 'delta_scaling_q975': 0.2879344500000003,
 'p_up_scaling': 0.16275,
 'lfsr_scaling': 0.16275,
 'delta_dev_median': -0.13729,
 'delta_dev_q025': -0.558748475,
 'delta_dev_q975': 0.31393597500000003,
 'p_up_dev': 0.27175,
 'lfsr_dev': 0.27175,
 'phi_median': 31.57465,
 'phi_q025': 28.796845,
 'phi_q975': 34.5889825,
 'b_noncancer_log_median': 7.75236,
 'b_noncancer_log_q025': 7.71813425,
 'b_noncancer_log_q975': 7.78327025,
 'b0_s1_median': 7.3156099999999995,
 'b0_s1_q025': 7.239029749999999,
 'b0_s1_q975': 7.386454,
 'b0_s2_median': 7.66358,
 'b0_s2_q

In [67]:
res_mcmc["ppc_summary"]

{'obs_mean': 2304.8519269776875,
 'obs_var': 398208.5547018667,
 'obs_zero_frac': 0.0,
 'ppc_mean_median': 2308.497971602434,
 'ppc_var_median': 406082.1682463115,
 'ppc_zero_frac_median': 0.0}

In [73]:
res_mcmc["ppc_draws"]

array([[2847., 2860., 1830., ..., 2478., 2773., 2496.],
       [3200., 2678., 2468., ..., 1926., 2239., 1636.],
       [2531., 3128., 2000., ..., 2959., 2921., 2340.],
       ...,
       [2206., 3820., 2659., ..., 2719., 1779., 1819.],
       [2526., 2831., 2619., ..., 2768., 2893., 2352.],
       [2992., 3273., 2579., ..., 2751., 3942., 1820.]])

In [39]:
# VI with mean-field Gaussian (fast)
res_vi = fit_one_gene_de(
    gene_df, 
    model,
    gene="BRAF",
    engine="vi_meanfield",
    vi_iter=20000,
    vi_output_samples=2000
)

11:37:37 - cmdstanpy - WARNING - Argument name `output_samples` is deprecated, please rename to `draws`.
11:37:37 - cmdstanpy - INFO - Chain [1] start processing
11:37:39 - cmdstanpy - INFO - Chain [1] done processing


In [41]:
res_vi

{'status': 'ok',
 'gene': 'BRAF',
 'N': 986,
 'n_aneup': 139,
 'cna': 'all',
 'subtype_levels': ['MSI', 'MSS'],
 'tumor0_lfc_median': 0.36813249358363676,
 'tumor0_lfc_q025': -0.5710830774500514,
 'tumor0_lfc_q975': 1.3014536454887646,
 'p_up_tumor': 0.7845,
 'lfsr_tumor': 0.21550000000000002,
 'p_rope_tumor': 0.3115,
 'delta_scaling_median': -0.4281595,
 'delta_scaling_q025': -1.0355795,
 'delta_scaling_q975': 0.1913634499999998,
 'p_up_scaling': 0.082,
 'lfsr_scaling': 0.082,
 'delta_dev_median': 0.03813785,
 'delta_dev_q025': -0.22434305,
 'delta_dev_q975': 0.31097547499999983,
 'p_up_dev': 0.6085,
 'lfsr_dev': 0.39149999999999996,
 'phi_median': 3.48994,
 'phi_q025': 3.2439105,
 'phi_q975': 3.7652379999999996,
 'b0_s1_median': 0.524419,
 'b0_s1_q025': -0.192360325,
 'b0_s1_q975': 1.2773764999999997,
 'b_scaling_s1_median': 0.19278,
 'b_scaling_s1_q025': -0.5584018999999999,
 'b_scaling_s1_q975': 0.980028125,
 'b_deviation_s1_median': -0.0479532,
 'b_deviation_s1_q025': -0.24258885,

#### Run on representative genes 

To plot prior vs posterior

In [41]:
#rep_genes = ["ZNRF3", "SMAD4", "APC", "LILRB3"]
rep_genes = ["ASXL1", "ERBB2", "SMAD4", "BCOR", "JUN"]

fits_rep = {}

for g in rep_genes:
    res = fit_one_gene_de(
        gene_df=gene_df,
        model=model,
        gene=g,
        engine="nuts",
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        adapt_delta=0.90,
        seed=123,
        save_draws=True,  
        output_dir = "/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/crc",
        show_progress=False,
        return_ppc=False
    )
    fits_rep[g] = res

17:35:44 - cmdstanpy - INFO - CmdStan start processing
17:35:44 - cmdstanpy - INFO - Chain [1] start processing
17:35:44 - cmdstanpy - INFO - Chain [2] start processing
17:35:44 - cmdstanpy - INFO - Chain [3] start processing
17:35:44 - cmdstanpy - INFO - Chain [4] start processing
17:37:01 - cmdstanpy - INFO - Chain [4] done processing
17:37:02 - cmdstanpy - INFO - Chain [2] done processing
17:37:02 - cmdstanpy - INFO - Chain [3] done processing
17:37:06 - cmdstanpy - INFO - Chain [1] done processing
17:37:28 - cmdstanpy - INFO - CmdStan start processing
17:37:28 - cmdstanpy - INFO - Chain [1] start processing
17:37:28 - cmdstanpy - INFO - Chain [2] start processing
17:37:28 - cmdstanpy - INFO - Chain [3] start processing
17:37:28 - cmdstanpy - INFO - Chain [4] start processing
18:21:54 - cmdstanpy - INFO - Chain [1] done processing
18:22:05 - cmdstanpy - INFO - Chain [4] done processing
18:22:06 - cmdstanpy - INFO - Chain [3] done processing
18:22:13 - cmdstanpy - INFO - Chain [2] do

In [47]:
fits_rep

{'ASXL1': {'status': 'ok',
  'gene': 'ASXL1',
  'N': 986,
  'n_aneup': 486,
  'cna': 'all',
  'subtype_levels': ['MSI', 'MSS'],
  'engine': 'nuts',
  'seed': 123,
  'output_dir': None,
  'tumor0_lfc_median': 0.6317792415259313,
  'tumor0_lfc_q025': 0.5024491691918245,
  'tumor0_lfc_q975': 0.7638433652320269,
  'p_up_tumor': 1.0,
  'lfsr_tumor': 0.0,
  'p_rope_tumor': 0.0,
  'delta_scaling_median': 0.01080435,
  'delta_scaling_q025': -0.59471285,
  'delta_scaling_q975': 0.6185814000000001,
  'p_up_scaling': 0.511,
  'lfsr_scaling': 0.489,
  'delta_dev_median': -0.299628,
  'delta_dev_q025': -0.7485315,
  'delta_dev_q975': 0.1443307,
  'p_up_dev': 0.0925,
  'lfsr_dev': 0.0925,
  'phi_median': 22.8,
  'phi_q025': 20.7840925,
  'phi_q975': 24.890617499999998,
  'b_noncancer_log_median': 8.279265,
  'b_noncancer_log_q025': 8.239113999999999,
  'b_noncancer_log_q975': 8.317952,
  'b0_s1_median': 7.598275,
  'b0_s1_q025': 7.5018955,
  'b0_s1_q975': 7.68991175,
  'b0_s2_median': 8.035325,
  'b

In [63]:
# Extract posterior draws and save them

rows = []

for g, res in fits_rep.items():
    draws_df = res["draws_subset"]
    if draws_df is None:
        continue

    for col in draws_df.columns:
        # keep only beta parameters
        if col.startswith("b_scaling") or col.startswith("b_deviation") or col.startswith("b0"):
            vals = draws_df[col].to_numpy()
            for v in vals:
                rows.append({
                    "gene": g,
                    "parameter": col,
                    "value": float(v)
                })

In [69]:
posterior_draws_df = pd.DataFrame(rows)
#posterior_draws_df.head()
posterior_draws_df.to_csv("/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/results/crc/genes_posterior_draws_v2.csv", index=False)

#### Parallel run on multiple genes

In [19]:
gene_groups = {g: gdf for g, gdf in df_long.groupby("gene", sort=False)}

def run_gene(g):
    return fit_one_gene_de(
        gene_groups[g],
        model,
        cna="all",
        chains=4,
        iter_warmup=500,
        iter_sampling=500,
        adapt_delta=0.90
    )

genes = list(gene_groups.keys())

results = Parallel(n_jobs=8, backend="loky")(
    delayed(run_gene)(g) for g in genes
)

res_df = pd.DataFrame(results)
res_df.head()

### Downstream results interpretation

In [95]:
@dataclass
class InterpretThresholds:
    # DE thresholds 
    de_lfsr_sig: float = 0.05        # call DE if lfsr <= this
    de_rope_high: float = 0.90       # call DE-null if p_rope >= this

    # Rewiring thresholds 
    rewire_lfsr_sig: float = 0.10    # call rewiring if lfsr <= this

    # --- Dosage class thresholds (per subtype) ---
    # We use posterior probabilities that our fit fn already computes.
    # "Sensitive" means CN perturbation yields effect beyond ROPE with high prob.
    dose_prob_sens: float = 0.80     # e.g. p_fracCN_2to3_pos >= 0.90 => sensitive to gain
    dose_prob_ins: float = 0.80      # e.g. p_fracCN_2to3_rope >= 0.90 => insensitive to gain

    # Dosage-compensation thresholds (if you output p_DC_gain_sX / p_DC_loss_sX)
    dc_prob: float = 0.80            # call compensated if >= this

    # If p_DC_* are not present, you can fall back to "cancel_index" if you output it:
    cancel_abs_rope: float = 0.20    # |cancel_index| <= this => strong cancellation (optional)

def _get(res: Dict[str, Any], key: str, default=None):
    return res.get(key, default)

def interpret_gene_result(
    res: Dict[str, Any],
    th: InterpretThresholds = InterpretThresholds(),
    assume_pairwise_s2_vs_s1: bool = True,
) -> Dict[str, Any]:
    """
    Interpret one gene's fitted result dict:
      - DE status (tumor baseline, subtype2 vs subtype1)
      - rewiring status (scaling / deviation)
      - dosage class per subtype (DSG / DIG / DCG) with optional gain/loss flags

    Returns a FLAT dict (good for DataFrame rows).
    """

    out: Dict[str, Any] = {
        "gene": _get(res, "gene"),
        "status": _get(res, "status"),
        "fit_flag": _get(res, "fit_flag", "ok"),
        "N": _get(res, "N"),
        "n_aneup": _get(res, "n_aneup"),
    }


    # 1) DE status (tumor baseline)

    lfsr_tumor = _get(res, "lfsr_tumor", np.nan)
    p_rope_tumor = _get(res, "p_rope_tumor", np.nan)

    if np.isfinite(lfsr_tumor) and lfsr_tumor <= th.de_lfsr_sig:
        de_status = "DE"
    elif np.isfinite(p_rope_tumor) and p_rope_tumor >= th.de_rope_high:
        de_status = "DE-null"
    else:
        de_status = "DE-uncertain"

    out.update({
        "de_status": de_status,
        "lfsr_tumor": lfsr_tumor,
        "p_rope_tumor": p_rope_tumor,
        # keep your effect size if present
        "tumor0_lfc_mean": _get(res, "tumor0_lfc_mean", np.nan),
        "tumor0_lfc_q025": _get(res, "tumor0_lfc_q025", np.nan),
        "tumor0_lfc_q975": _get(res, "tumor0_lfc_q975", np.nan),
    })

  
    # 2) Rewiring status (between subtypes)
    
    lfsr_scaling = _get(res, "lfsr_scaling", np.nan)
    lfsr_dev = _get(res, "lfsr_dev", np.nan)

    scaling_rewired = (np.isfinite(lfsr_scaling) and lfsr_scaling <= th.rewire_lfsr_sig)
    dev_rewired     = (np.isfinite(lfsr_dev) and lfsr_dev <= th.rewire_lfsr_sig)

    if scaling_rewired and dev_rewired:
        rewiring = "rewired:scaling+deviation"
    elif scaling_rewired:
        rewiring = "rewired:scaling"
    elif dev_rewired:
        rewiring = "rewired:deviation"
    else:
        rewiring = "not_rewired"

    out.update({
        "rewiring_status": rewiring,
        "lfsr_scaling": lfsr_scaling,
        "lfsr_dev": lfsr_dev,
        "delta_scaling_mean": _get(res, "delta_scaling_mean", np.nan),
        "delta_scaling_q025": _get(res, "delta_scaling_q025", np.nan),
        "delta_scaling_q975": _get(res, "delta_scaling_q975", np.nan),
        "delta_dev_mean": _get(res, "delta_dev_mean", np.nan),
        "delta_dev_q025": _get(res, "delta_dev_q025", np.nan),
        "delta_dev_q975": _get(res, "delta_dev_q975", np.nan),
    })

    
    # 3) Dosage class per subtype
    
    subtype_levels = _get(res, "subtype_levels", None)  # e.g. ["MSI","MSS"]
    if subtype_levels is None:
        # still proceed with s1,s2,... keys
        subtype_levels = []

    # infer how many subtypes are present from outputs (robust)
    # Prefer explicit list; else scan for p_fracCN_2to3_* keys
    S = len(subtype_levels)
    if S == 0:
        # detect max s index present
        s_candidates = []
        for k in res.keys():
            if k.startswith("p_fracCN_2to3_pos_s"):
                try:
                    s_candidates.append(int(k.split("_s")[-1]))
                except Exception:
                    pass
        S = max(s_candidates) if s_candidates else 2  # fallback

    out["S"] = S
    if subtype_levels:
        out["subtype_levels_str"] = "|".join(map(str, subtype_levels))

    def subtype_name(s: int) -> str:
        if 1 <= s <= len(subtype_levels):
            return str(subtype_levels[s-1])
        return f"s{s}"

    # helper for dosage decision per subtype
    def dosage_class_for_subtype(s: int) -> Dict[str, Any]:
        name = subtype_name(s)

        # Evidence for being sensitive vs insensitive for gain (2->3) and loss (2->1)
        p_gain_pos  = _get(res, f"p_fracCN_2to3_pos_s{s}", np.nan)
        p_gain_rope = _get(res, f"p_fracCN_2to3_rope_s{s}", np.nan)
        p_loss_neg  = _get(res, f"p_fracCN_2to1_neg_s{s}", np.nan)
        p_loss_rope = _get(res, f"p_fracCN_2to1_rope_s{s}", np.nan)

        # Optional DC outputs
        p_dc_gain = _get(res, f"p_DC_gain_s{s}", np.nan)
        p_dc_loss = _get(res, f"p_DC_loss_s{s}", np.nan)

        # Optional cancellation index (if you output it)
        cancel_gain_mean = _get(res, f"cancel_index_2to3_s{s}_mean", np.nan)
        cancel_loss_mean = _get(res, f"cancel_index_2to1_s{s}_mean", np.nan)

        # classify gain/loss directionally (useful diagnostics)
        gain_flag = "unknown"
        if np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens:
            gain_flag = "sensitive"
        elif np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins:
            gain_flag = "insensitive"

        loss_flag = "unknown"
        if np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens:
            loss_flag = "sensitive"
        elif np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins:
            loss_flag = "insensitive"

        # primary subtype dosage class
        # Priority: DC > DIS > DS > uncertain
        dc_gain = (np.isfinite(p_dc_gain) and p_dc_gain >= th.dc_prob)
        dc_loss = (np.isfinite(p_dc_loss) and p_dc_loss >= th.dc_prob)

        # Fallback DC using cancellation index if p_DC_* missing
        if (not np.isfinite(p_dc_gain)) and np.isfinite(cancel_gain_mean):
            dc_gain = (abs(cancel_gain_mean) <= th.cancel_abs_rope)
        if (not np.isfinite(p_dc_loss)) and np.isfinite(cancel_loss_mean):
            dc_loss = (abs(cancel_loss_mean) <= th.cancel_abs_rope)

        any_dc = dc_gain or dc_loss

        # DIS if both gain and loss are ROPE-mostly (or at least gain ROPE-mostly and loss ROPE-mostly)
        dis_gain = (np.isfinite(p_gain_rope) and p_gain_rope >= th.dose_prob_ins)
        dis_loss = (np.isfinite(p_loss_rope) and p_loss_rope >= th.dose_prob_ins)
        is_dis = dis_gain and dis_loss

        # DS if strong sensitivity at least on one side (commonly gain), ideally both
        ds_gain = (np.isfinite(p_gain_pos) and p_gain_pos >= th.dose_prob_sens)
        ds_loss = (np.isfinite(p_loss_neg) and p_loss_neg >= th.dose_prob_sens)
        is_ds = ds_gain or ds_loss

        if any_dc:
            cls = "DCG"   # dosage-compensated
        elif is_dis:
            cls = "DIG"  # dosage-insensitive
        elif is_ds:
            cls = "DSG"   # dosage-sensitive
        else:
            cls = "UNC"

        # attach a few useful summary numbers if present
        return {
            f"dosage_class_{name}": cls,
            f"gain_flag_{name}": gain_flag,
            f"loss_flag_{name}": loss_flag,
            f"p_gain_pos_{name}": p_gain_pos,
            f"p_gain_rope_{name}": p_gain_rope,
            f"p_loss_neg_{name}": p_loss_neg,
            f"p_loss_rope_{name}": p_loss_rope,
            f"p_DC_gain_{name}": p_dc_gain,
            f"p_DC_loss_{name}": p_dc_loss,
        }

    for s in range(1, S + 1):
        out.update(dosage_class_for_subtype(s))

    # 4) Optional combined label for easy plotting
    
    # Example: "DE-null | not_rewired | MSI:DS, MSS:DS"
    per_sub = []
    for s in range(1, S + 1):
        name = subtype_name(s)
        per_sub.append(f"{name}:{out.get(f'dosage_class_{name}', 'NA')}")
    out["summary_label"] = f"{de_status} | {rewiring} | " + ",".join(per_sub)

    return out

#### Load model fit results

In [1]:
DATA_PATH = "/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/results/crc"
res_crc = pd.read_csv(os.path.join(DATA_PATH, "crc_res_nb_de_drivers.csv"))
res_crc.head()

In [25]:
res_crc = res_crc.drop(
    columns=['Unnamed: 0.1', 'Unnamed: 0', 'Unnamed: 0.2', 'status', 'success', 'gene.1']
)
res_crc.head()

,gene,error,N,n_aneup,cna,subtype_levels,tumor0_lfc_median,tumor0_lfc_q025,tumor0_lfc_q975,p_up_tumor,...,lp_dev_2to4_s2_q025,lp_dev_2to4_s2_q975,Rhat_phi,ess_phi,max_Rhat_core,min_ESS_core,n_divergent,max_treedepth,n_max_treedepth,fit_flag
0,ACVR1B,NaN,986,39,all,"['MSI', 'MSS']",0.968467,0.826774,1.106175,1.00000,...,-0.676427,0.663717,1.000660,4875.94,1.00287,3044.58,0,7,0,ok
1,ACVR2A,NaN,986,18,all,"['MSI', 'MSS']",0.131804,-0.000210,0.264902,0.97475,...,-0.387237,1.105975,1.001510,4979.91,1.00284,3100.91,0,7,0,ok
2,AKT1,NaN,986,95,all,"['MSI', 'MSS']",-0.398087,-0.473976,-0.318741,0.00000,...,0.118539,1.372487,0.999876,4871.14,1.00102,2919.69,0,7,0,ok
3,AMER1,NaN,986,173,all,"['MSI', 'MSS']",0.868841,0.722534,1.027364,1.00000,...,0.530408,1.453063,1.000880,4750.93,1.00235,3262.13,0,7,0,ok
4,ANKRD40,NaN,986,50,all,"['MSI', 'MSS']",0.121918,0.029585,0.215858,0.99600,...,-0.085322,0.543418,0.999606,4151.13,1.00365,2659.27,0,7,0,ok


In [13]:
res_crc["result"] = res_crc["result"].apply(ast.literal_eval)
result_df = pd.json_normalize(res_crc["result"])
df_expanded = pd.concat([res_crc.drop(columns=["result"]), result_df], axis=1)
df_expanded.head()

,Unnamed: 0.1,Unnamed: 0,gene,success,error,status,gene,N,n_aneup,cna,...,lp_dev_2to4_s2_q025,lp_dev_2to4_s2_q975,Rhat_phi,ess_phi,max_Rhat_core,min_ESS_core,n_divergent,max_treedepth,n_max_treedepth,fit_flag
0,0,0,ACVR1B,True,NaN,ok,ACVR1B,986,39,all,...,-0.676427,0.663717,1.000660,4875.94,1.00287,3044.58,0,7,0,ok
1,1,1,ACVR2A,True,NaN,ok,ACVR2A,986,18,all,...,-0.387237,1.105975,1.001510,4979.91,1.00284,3100.91,0,7,0,ok
2,2,2,AKT1,True,NaN,ok,AKT1,986,95,all,...,0.118539,1.372487,0.999876,4871.14,1.00102,2919.69,0,7,0,ok
3,3,3,AMER1,True,NaN,ok,AMER1,986,173,all,...,0.530408,1.453063,1.000880,4750.93,1.00235,3262.13,0,7,0,ok
4,4,4,ANKRD40,True,NaN,ok,ANKRD40,986,50,all,...,-0.085322,0.543418,0.999606,4151.13,1.00365,2659.27,0,7,0,ok


In [27]:
res_crc.to_csv("/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/results/crc/crc_res_nb_de_drivers.csv", index=True)

#### Results interpreter

In [105]:
filtered = res_drivers[res_drivers["fit_flag"] == "ok"]

interpreter = [
    interpret_gene_result(r)
    for r in filtered.to_dict("records")
]

#### Parameters posterior draws

In [71]:
def load_selected_gene_draws(fit_root, genes, subtype_labels=None):
    """
    Load draws_subset.npz for selected genes and reshape to tidy format.

    Output columns:
        gene | draw | param | subtype | value
    """

    fit_root = Path(fit_root)
    rows = []

    for gene in genes:

        npz_file = fit_root / gene / "draws_subset.npz"
        if not npz_file.exists():
            print(f"Skipping {gene}: file not found")
            continue

        data = np.load(npz_file, allow_pickle=True)

        for param in data.files:
            arr = data[param]

            # Global parameters
            if arr.ndim == 1:
                for i, v in enumerate(arr):
                    rows.append({
                        "gene": gene,
                        "draw": i + 1,
                        "param": param,
                        "subtype": None,
                        "value": float(v),
                    })

            # Subtype-specific parameters
            elif arr.ndim == 2:
                n_draws, S = arr.shape

                for s in range(S):
                    subtype = (
                        subtype_labels[s]
                        if subtype_labels is not None
                        else f"s{s+1}"
                    )

                    for i in range(n_draws):
                        rows.append({
                            "gene": gene,
                            "draw": i + 1,
                            "param": param,
                            "subtype": subtype,
                            "value": float(arr[i, s]),
                        })

    df = pd.DataFrame(rows)
    return df

In [79]:
genes = ["ASXL1", "ERBB2", "SMAD4", "BCOR", "JUN"]

posterior_df = load_selected_gene_draws(
    fit_root="/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/results/crc/fits_nb_de",
    genes=genes,
    subtype_labels=["MSI", "MSS"]
)

posterior_df = posterior_df[
    posterior_df["param"].isin(["b0", "b_scaling", "b_deviation"])
]
posterior_df

,gene,draw,param,subtype,value
20000,ASXL1,1,b0,MSI,7.680339
20001,ASXL1,2,b0,MSI,7.630135
20002,ASXL1,3,b0,MSI,7.569186
20003,ASXL1,4,b0,MSI,7.527035
20004,ASXL1,5,b0,MSI,7.636168
...,...,...,...,...,...
219995,JUN,3996,b_deviation,MSS,0.223367
219996,JUN,3997,b_deviation,MSS,-0.071361
219997,JUN,3998,b_deviation,MSS,0.356837
219998,JUN,3999,b_deviation,MSS,0.500328


In [81]:
print(posterior_df["param"].unique())

['b0' 'b_scaling' 'b_deviation']


In [83]:
posterior_df.to_csv("/Users/katsiarynadavydzenka/Documents/PhD_AI/nb_stan/results/crc/genes_posterior_draws_v2.csv", index=False)